# Marmousi2 Acoustic bv1.2 Forward Modeling Validation

This notebook validates the forward-modeling side of the Marmousi2 acoustic case. It does not run inversion and does not modify the original Marmousi2 notebooks.

## Scope

- `check`: rebuild model, survey, observed data, and backend objects.
- `forward`: run one true-model single-shot forward modeling check.

Use this notebook before running inversion validation.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "ADFWI").exists():
    REPO_ROOT = Path("/liufeng1afs/project/04_Inversion/ADFWI-github")

SCRIPT = REPO_ROOT / "examples" / "validation" / "marmousi2_acoustic_bv12" / "scripts" / "run_validation.py"
OUTPUT_ROOT = REPO_ROOT / "examples" / "validation" / "marmousi2_acoustic_bv12" / "outputs"
SCRIPT


In [ ]:
def run_stage(stage: str, *, dry_run: bool = True, overwrite: bool = False, device: str = "npu:0", extra_args: list[str] | None = None):
    command = [sys.executable, str(SCRIPT), stage, "--device", device, "--output-root", str(OUTPUT_ROOT)]
    if dry_run:
        command.append("--dry-run")
    if overwrite:
        command.append("--overwrite")
    if extra_args:
        command.extend(extra_args)
    proc = subprocess.run(command, cwd=str(REPO_ROOT), text=True, capture_output=True, check=False)
    if proc.stderr:
        print(proc.stderr)
    if proc.stdout:
        print(proc.stdout)
    if proc.returncode != 0:
        raise RuntimeError(f"stage {stage} failed with return code {proc.returncode}")
    return json.loads(proc.stdout[proc.stdout.find("{"):])


## Preview Forward Commands

This dry run prints the commands without running the propagator.

In [ ]:
plan = run_stage("forward", dry_run=True, device="npu:0")
plan["stages"][0]["command_text"]

## Run Case Check

Set `RUN_FORWARD_MODELING = True` to execute the read-only check and the single-shot forward check.

In [ ]:
RUN_FORWARD_MODELING = False

if RUN_FORWARD_MODELING:
    check_result = run_stage("check", dry_run=False, overwrite=True, device="npu:0")
    forward_result = run_stage("forward", dry_run=False, overwrite=True, device="npu:0")
else:
    check_result = {"status": "skipped", "reason": "set RUN_FORWARD_MODELING=True"}
    forward_result = {"status": "skipped", "reason": "set RUN_FORWARD_MODELING=True"}

check_result, forward_result

## Inspect Outputs

Forward outputs are written under `outputs/check_*` and `outputs/forward_*`. Inspect `summary.json`, `stdout.txt`, and `stderr.txt`.